# 04 · Retrieval — LanceDB + nv-embedqa + nv-rerankqa

**Purpose** — build the semantic index and show reranking earning its place.

**Prerequisites** — 01 has been run. Embedding and reranking endpoints reachable.

### Scope
Most manager questions ("what is the state of RO-x", "which are blocked",
"what did EMP014 do") are **structured** queries answered by the deterministic
tools — not by retrieval. This serves the narrative slice: *has anyone seen this
fault before*, *what did the last technician say*.

Building the index is a one-off batch job; only the query embedding is per-call.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))          # notebooks/ -> repo root
os.chdir(os.path.abspath(".."))
from dotenv import load_dotenv; load_dotenv()
print("repo root:", os.getcwd())

## Execute — build the index (one pass over every update)

In [ ]:
from app.retrieval.index import build
stats = build()
print(stats)

## What you should see — search, then rerank

In [ ]:
from app.retrieval.index import search
Q = "grinding noise from the front brakes under braking"
hits = search(Q, k=8, rerank_to=4)
for i, h in enumerate(hits, 1):
    print(f"{i}. {h['ro_number']}  {h.get('vehicle')}  [{h.get('rerank_score', h.get('vector_score')):.3f}]")
    print(f"   {h['text'][:150]}")

## Reranking changes the order

Vector search retrieves broadly; the reranker reads the query against each
passage properly. Compare the top 4 with and without it.

In [ ]:
raw = search(Q, k=6, rerank_to=None)
rer = search(Q, k=6, rerank_to=4)
print("vector only :", [h["ro_number"] for h in raw[:4]])
print("reranked    :", [h["ro_number"] for h in rer])

## Agent-tool shape — passages with citations

In [ ]:
from app.retrieval.index import search_updates
r = search_updates("battery keeps going flat overnight", k=3)
print("citations:", r["citations"])
for p in r["passages"]:
    print(f"  [{p['update_id']}] {p['ro_number']} {p['by']}: {p['text'][:110]}")